# 03 Feature Engineering

**03 Notebook will house all our feature engineering efforts.**

+ Feature brainstorming
+ Sanity checks
+ Distribution checks
+ General dataset behavioral characteristic/trend analysis

In [2]:
%load_ext autoreload
%autoreload 2

import sys 
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

sys.path.insert(0, '../src')

from icu_tft.data.connect import get_connection
from icu_tft.data.static_features import (
    build_static_features,
    feature_summary,
    FEATURE_GROUPS,
)

from icu_tft.data.extract_timeseries import (
    build_timeseries,
    validate_cohort,
    ALL_FEATURES,
)

logging.basicConfig(
    level=logging.INFO,
    format='%(acstime)s %(name)s %(levelname)s %(message)s',
    datefmt='%H:%M:%S',
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi':130, 'savefig.dpi':300})

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# cds

PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('reports/figures')
COHORT_PQ = PROCESSED_DIR / 'cohort.parquet'
STATIC_PQ = PROCESSED_DIR / 'static_features.parquet'
TS_PQ = PROCESSED_DIR / 'timeseries.parquet'
FINAL_PQ = PROCESSED_DIR / 'final_static_df.parquet'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('paths ok:')
print(f'cohort  : {COHORT_PQ}')
print(f'static : {STATIC_PQ} ')
print(f'ts : {TS_PQ}')
print(f'final : {FINAL_PQ} ')


paths ok:
cohort  : ../data/processed/cohort.parquet
static : ../data/processed/static_features.parquet 
ts : ../data/processed/timeseries.parquet
final : ../data/processed/final_static_df.parquet 


# Cohort Loading and registering DuckDB views

In [4]:
con = get_connection()

assert COHORT_PQ.exists(), (
    f'cohort.parquet not found at {COHORT_PQ}.\n'
    f'Run notebook 91 that extracts cohort first. '
)

[connect.py] Registered 30 views against mimic.duckdb
[connect.py] WARNING — 1 file(s) not found (views skipped):
  /Users/longer/ICU_Mortality_Prediction/data/raw/hosp/antimicrobial.csv.gz


In [5]:
cohort = pd.read_parquet(COHORT_PQ)

print(f' cohort loaded. {cohort.shape[0]:,} stays x {cohort.shape[1]} columns')
print(f' columns: {list(cohort.columns)}')
print('===' * 30)
print('Target Label Distribution (mortality_24h)')
vc = cohort['mortality_24h'].value_counts()
for label, count in vc.items():
    print(f'    {label}: {count:,} ({count / len(cohort):.1%})')

 cohort loaded. 67,223 stays x 14 columns
 columns: ['subject_id', 'hadm_id', 'stay_id', 'gender', 'anchor_age', 'admission_type', 'first_careunit', 'icu_los_hours', 'hospital_los_hours', 'mortality_24h', 'mortality_inhospital', 'insurance', 'race', 'marital_status']
Target Label Distribution (mortality_24h)
    0: 66,472 (98.9%)
    1: 751 (1.1%)


# Static Features

In [6]:
import icu_tft.data.static_features as _sf
import types

# types package needed for fixing up one of the import functions

def _extract_severity_proxies_updated(cohort: pd.DataFrame, con) -> pd.DataFrame:
    '''updated and optimized version of the severity proxy extraction function.'''
    stay_ids = tuple(cohort['stay_id'].unique().tolist())
    
    query = f'''
        SELECT 
            ce.stay_id, 
            MIN(CASE WHEN ce.itemid = 220052 THEN ce.valuenum END) AS severity_min_map_6h,
            MAX(CASE WHEN ce.itemid = 50813  THEN ce.valuenum END) AS severity_max_lactate_6h
        FROM mimic_icu.chartevents ce
        INNER JOIN mimic_icu.icustays ie ON ce.stay_id = ie.stay_id
        WHERE ce.stay_id IN {stay_ids}
            AND ce.itemid IN (220052, 50813)
            AND ce.charttime >= ie.intime
            AND ce.charttime <= ie.intime + INTERVAL '6 hours'
            AND ce.valuenum IS NOT NULL
        GROUP BY ce.stay_id
    '''
    proxies_df = con.execute(query).df()
    
    gcs_query = f'''
        WITH gcs_components AS (
            SELECT 
                ce.stay_id, 
                ce.charttime,
                SUM(ce.valuenum) AS gcs_total
            FROM mimic_icu.chartevents ce
            INNER JOIN mimic_icu.icustays ie ON ce.stay_id = ie.stay_id
            WHERE ce.stay_id IN {stay_ids}
                AND ce.itemid IN (223900, 223901, 220739)
                AND ce.charttime >= ie.intime 
                AND ce.charttime <= ie.intime + INTERVAL '6 hours'
            GROUP BY ce.stay_id, ce.charttime
            HAVING COUNT(DISTINCT ce.itemid) = 3
        )
        SELECT 
            stay_id, 
            MIN(gcs_total) AS severity_min_gcs_6h
        FROM gcs_components
        GROUP BY stay_id
    '''
    gcs_df = con.execute(gcs_query).df()


    merged = pd.merge(cohort[['stay_id']], proxies_df, on='stay_id', how='left')
    merged = pd.merge(merged, gcs_df, on='stay_id', how='left')
    return merged

_sf.extract_severity_proxies = _extract_severity_proxies_updated
print('Severity proxy function patched (gcs_query fix applied).')
    

Severity proxy function patched (gcs_query fix applied).
